# Analysis & reporting

Select a **single culture** or a **group of cultures** (from the same experiment or across
experiments) and generate an MXtreme PDF report. Cultures are selected against the managed-store
registry via `CultureID` / `CultureSelector`; `mxtreme.analysis.generate_report` resolves the
selection, runs the requested sections, and writes one PDF.

This assumes the store already holds preprocessed `.npz` files and burst CSVs (see
`preprocess_pipeline.ipynb` and `burst_detection.ipynb`).

In [ ]:
from mxtreme.config import Config
from mxtreme.identity import CultureID, CultureSelector
from mxtreme.phases import phases_from_event_tags
from mxtreme.analysis import generate_report

config = Config.from_toml("mxtreme.toml")
print("managed store:", config.data_root.resolve())
print("analysis dir: ", config.analysis_dir.resolve())

## Phases (optional but recommended for phased experiments)

Per-phase activity/burst windows are computed from a recording's **phases**. Burst CSVs already
carry a `phase` label from detection time; to make the analysis reconstruct the same phase windows
(so burst-rate denominators are correct), pass a `make_phases` callable that rebuilds them from the
maxlab event tags. Omit `make_phases` for unphased data — every recording then has a single
`"full"` phase.

The `burstTrainer` example recordings tag `pre_recording_start` / `closed_loop_start` /
`post_recording_start`, matching what `burst_detection.ipynb` used.

In [ ]:
PHASE_TAGS = [
    ("pre",   "pre_recording_start"),
    ("train", "closed_loop_start"),
    ("post",  "post_recording_start"),
]
END_TAG = "end_experiment"

def make_phases(rec):
    end = int(rec.spike_data["frameno"].max())
    return phases_from_event_tags(rec.event_df, PHASE_TAGS, end_frame=end, end_tag=END_TAG)

## Report sections

`sections` selects what goes in the PDF:

- `"overview"`    — single culture: ASDR + MEA layout per DIV; group: a concise summary table.
- `"activity"`    — firing rate / ISI / spike amplitude / active channels.
- `"bursting"`    — detection diagnostics + burst stats (IBI, rate, size, duration).
- `"stimulation"` — total stim/train time per DIV (auto-skips non-stim cultures).
- `"performance"` — a learning curve from a pluggable objective (default: burst direction).

The default is `("overview", "activity", "bursting")`.

In [ ]:
SECTIONS = ("overview", "activity", "bursting", "stimulation", "performance")

## 1. A single culture

A `CultureID(exp_id, chip, well)` expands to all completed DIVs for that culture. The overview
shows the ASDR and MEA layout.

In [ ]:
single = CultureID("burstTrainer", "M07140", "0")

report_path = generate_report(
    single, config,
    sections=SECTIONS,
    make_phases=make_phases,
)
report_path

## 2. A group from the same experiment

Two or more wells of the same MaxTwo chip. A `CultureSelector` yields population-level (mean ± SEM
across cultures) activity and burst panels, and the overview becomes a concise table.

In [ ]:
group_same_exp = CultureSelector(cultures=[
    CultureID("burstTrainer", "M07140", "0"),
    CultureID("burstTrainer", "M07140", "1"),
    CultureID("burstTrainer", "M07140", "2"),
])

generate_report(group_same_exp, config, sections=SECTIONS, make_phases=make_phases)

## 3. A group across experiments (MaxOne + MaxTwo)

Cultures from different experiments — here a MaxTwo well (`M07140`) and a MaxOne well
(`P004722`). `resolve_paths` groups them by experiment; the report pools them into one population
overview.

In [ ]:
group_cross_exp = CultureSelector(cultures=[
    CultureID("burstTrainer",   "M07140",   "0"),   # MaxTwo
    CultureID("m1BurstTrainer", "P004722", "0"),   # MaxOne
])

generate_report(group_cross_exp, config, sections=SECTIONS, make_phases=make_phases)

## Custom performance objective

The performance section's objective is pluggable: pass `objective_fn(burst_df) -> float`. The
default scores burst propagation direction (fraction with `origin_x < peak_x`). Supply your own for
a task-specific readout — e.g. mean burst size as a crude excitability proxy:

In [ ]:
def mean_size_objective(burst_df):
    return float(burst_df["size_frac_elec"].mean()) if len(burst_df) else float("nan")

generate_report(
    single, config,
    sections=("performance",),
    make_phases=make_phases,
    objective_fn=mean_size_objective,
    output_path=config.analysis_dir / "reports" / "M07140_well0_custom_objective.pdf",
)